# Transport: selection diagrams

A selection diagram is the causal graph plus S-nodes on the mechanisms that differ between
the study population and the target. `CausalGraph.selection` lists those nodes;
`selection_diagram` makes the S-nodes explicit. Transportability is then d-separation
(Bareinboim & Pearl 2014): the effect transports *directly* if $Y \perp S \mid X$ in
$G_{\overline{X}}$, and via the transport formula
$P^*(y \mid do(x)) = \sum_z P(y \mid do(x), z)\,P^*(z)$ if some $Z$ is S-admissible.

In [ ]:
import numpy as np

from axiom.identify import (
    CausalGraph, TransportVerdict, directly_transportable, identify, minimal_s_admissible_sets,
    ols, s_admissible, s_admissible_sets, selection_diagram, transport_verdict,
    trivially_transportable,
)
from axiom.sim import transport_pair

from axiom.display import enable

enable();  # every axiom result renders itself from here on

In [ ]:
g = CausalGraph.from_edges("Z -> X, Z -> Y, X -> Y", selection=["Z"])
sd = selection_diagram(g)
print(sd.nodes, "| unmeasured:", sd.unmeasured)
print("direct:", directly_transportable(g, "X", "Y"))
print("S-admissible {Z}:", s_admissible(g, "X", "Y", ["Z"]), "| {}:", s_admissible(g, "X", "Y", []))
print(s_admissible_sets(g, "X", "Y"), minimal_s_admissible_sets(g, "X", "Y"))
print("trivially (from target data alone):", trivially_transportable(g, "X", "Y"))

## Verdicts

`transport_verdict` searches for a route; pass `given=` to check a specific set — the empty
set is "read the source effect as the target's, unadjusted", which the graph here blocks.
When no sufficient condition the module implements holds, the verdict is `unsupported`
(the complete sID algorithm is out of scope for 1.0), never a false `blocked`.

In [ ]:
unadjusted: TransportVerdict = transport_verdict(g, "X", "Y", given=())
print(unadjusted.verdict.status, "|", unadjusted.verdict.reason)
adjusted = transport_verdict(g, "X", "Y")
print(adjusted.verdict.status, adjusted.route, adjusted.s_admissible_set, "|", adjusted.formula)
print([f"{a.name}:{a.state}" for a in adjusted.verdict.assumptions])

In [ ]:
cases = {
    "S only on X": CausalGraph.from_edges("Z -> X, Z -> Y, X -> Y", selection=["X"]),
    "S on Y, back-door in target": CausalGraph.from_edges("Z -> X, Z -> Y, X -> Y", selection=["Y"]),
    "S on mediator": CausalGraph.from_edges("X -> M, M -> Y", selection=["M"]),
    "no S-nodes": CausalGraph.from_edges("X -> Y"),
}
for label, graph in cases.items():
    tv = transport_verdict(graph, "X", "Y")
    print(f"{label:28s} {tv.verdict.status:11s} {tv.route:26s} {tv.formula}")

`identify()` attaches the transport verdict whenever the graph has S-nodes.

In [ ]:
v = identify(g, "X", "Y")
print(v.route, v.status, "| transport:", v.transport.verdict.status if v.transport else None)

## Recovery on a pair of worlds

`transport_pair()` gives two linear worlds differing only in the distribution of `Z`. The
slope of `X` is invariant; the *level* of the effect is not. Applying the transport formula
(fit in the source, standardize over the target's `Z`) recovers the target truth; reading the
source number as the target's misses by exactly the amount the formula predicts.

In [ ]:
source, target = transport_pair()
s = source.observed(source.simulate(40_000, seed=0))
t = target.observed(target.simulate(40_000, seed=1))
x0 = 1.0
truth = target.interventional_mean("Y", intervene={"X": x0})

design = np.column_stack([np.ones(len(s)), s["X"], s["Z"]])
a, b_x, b_z = np.linalg.lstsq(design, s["Y"].to_numpy(), rcond=None)[0]
transported = a + b_x * x0 + b_z * t["Z"].mean()
untransported = a + b_x * x0 + b_z * s["Z"].mean()
print(f"target truth E*[Y|do(X=1)] = {truth:.3f}")
print(f"transported {transported:.3f} | untransported {untransported:.3f} | predicted gap {b_z * (t['Z'].mean() - s['Z'].mean()):.3f}")
print("slope estimate in the source:", ols(s, "Y", "X", covariates=["Z"]).estimate.__round__(3))